In [1]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from datetime import datetime
from datetime import timedelta
from openpyxl.utils import get_column_letter
import datetime  
import pdb
import pandas as pd
import numpy as np 
# from pandas.io.json import json_normalize  
from pandas import json_normalize
import json
import psycopg2
from datetime import datetime
from dateutil.relativedelta import relativedelta
from urllib.parse import quote
from sqlalchemy.engine import create_engine
import warnings
from pandas import Timestamp
import dateutil
import pdb
import datetime

# import mysql.connector
import pandas as pd
import numpy as np
from numpy import nan
import ast
from pandas import Timestamp
import pickle
import joblib
import openpyxl
from openpyxl import load_workbook
from openpyxl.utils import FORMULAE
import dateutil
import pdb
import datetime
import pdb
import pandas as pd
import numpy as np 
# from pandas.io.json import json_normalize  
import json
import psycopg2
from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta
from urllib.parse import quote
from sqlalchemy.engine import create_engine
import warnings
warnings.filterwarnings("ignore")

from openpyxl import Workbook
from openpyxl.utils import get_column_letter
import pdb

warnings.filterwarnings("ignore")
# C:\Users\saurabh.khopkar\Saurabh pythond code save path


In [2]:
# df0 = pd.read_excel(r"C:\Users\saurabh.khopkar\Downloads\Trend_Login_Disb_Base.xlsx", sheet_name="Disb") #reads the first sheet of your excel file


In [3]:
# sas_config = r"C:\Users\mononita.mitra\OneDrive - UGRO Capital\Desktop\Mononita Data 14-09-2023\Mononita\GroMicro Login Analysis\config_ods.json"
sas_config = r"C:\Users\saurabh.khopkar\Downloads\config_ods_Saurabh.json"
# sas_config = r"C:\Users\saurabh.khopkar\Downloads\config_ods 2.json"
def connection_from_sas():
    try:
        f = open(sas_config)
        data = json.load(f)

 

        database = data.get("database")
        user = data.get("user")
        password = data.get("password")
        host = data.get("host")
        port = data.get("port")

 

        connection = psycopg2.connect(database=database, user=user, password=password, host=host, port=port)
        return connection

 

    except Exception as error:
        print(error)

In [4]:
connection = connection_from_sas()

In [5]:
sql_query_1 = """
select laf,gender_value,buss_pan,
case 
when buss_pan like '___P%'  then 'Proprietorship'
when buss_pan like '___C%'  then 'Company'
when buss_pan like '___T%'  then 'Turst'
when buss_pan like '___H%'  then 'HUF'
when buss_pan like '___F%'  then 'Partnership Firm'
when buss_pan like '___A%'  then 'Educational Institute'
when buss_pan like '___J%'  then 'Charity'
when buss_pan like '___L%'  then 'Local Authority' 
else 'NA'
end as PAN_constitution
from tb_bureau_data tbd 
where 
gender_value is not null
and edl_created_at >='2025-11-04'
order by gender_value 
"""
# where laf ='ANT22050401400555833'
# cast(pa_first_decision as date) as pa_first_decision,
# select object_pri_key_1,equipment_make,equipment_type from collateral where equipment_make <>''
# and object_pri_key_1 in ('4FIN22050203550050840','4FIN220623033842642227','4FIN220720022010710305')
# where b.datetime_login >='2024-09-01'
# left join laf_detail_status b on a.object_pri_key_1=b.laf_number
# where b.datetime_login >='2024-09-01'

In [6]:
df01 = pd.read_sql_query(sql_query_1, connection)

In [7]:
df01.columns

Index(['laf', 'gender_value', 'buss_pan', 'pan_constitution'], dtype='object')

In [8]:
df1 = df01.groupby('laf').agg({'gender_value': ','.join
                             }).reset_index()

In [9]:
df1.columns

Index(['laf', 'gender_value'], dtype='object')

In [10]:
df01.columns

Index(['laf', 'gender_value', 'buss_pan', 'pan_constitution'], dtype='object')

In [11]:
neworder = ['laf','pan_constitution']
df01=df01.reindex(columns=neworder)

In [12]:
df01 = df01.sort_values("laf").drop_duplicates(["laf"], keep="first")

In [13]:
df1=pd.merge(df1,df01,on='laf',how='left')

In [14]:
df1.columns

Index(['laf', 'gender_value', 'pan_constitution'], dtype='object')

In [15]:
df1['gender_value'] = df1['gender_value'].str.capitalize()


In [16]:
df161 = df1[(df1["laf"]=="LCHU022520191221123639")] #Filtering dataframe
# df161 = df2[(df16['Foreclosure_LOB'].isin(['NA'])] #Filtering dataframe
neworder = ['laf','gender_value']
df162=df161.reindex(columns=neworder)
df162

,laf,gender_value
86875,LCHU022520191221123639,"Female,female,female,female,male,male,male,male"


In [17]:
df1['left_2'] = [x[:2] for x in df1['gender_value']]


In [18]:
df1

,laf,gender_value,pan_constitution,left_2
0,\nRUPITOL24092302300292772,"Female,female,female,female,female,female,fema...",Proprietorship,Fe
1,\nRUPITOL241011054500100189,"Female,female,female,female,male,male,male,male",Proprietorship,Fe
2,\rRUPITOL241014111500100476,"Female,female,female,female,male,male,male,male",Proprietorship,Fe
3,4FIN22041903450040945,"Female,female,female,female",Proprietorship,Fe
4,4FIN22050203550050840,"Female,female,female,female",Proprietorship,Fe
...,...,...,...,...
466255,ZIP220129113034134166,"Female,female,female,female,male,male,male,male",Proprietorship,Fe
466256,ZIP220129113039139169,"Male,male,male,male,male,male,male,male",Proprietorship,Ma
466257,ZIP220129113044144172,"Female,female,female,female,male,male,male,male",Proprietorship,Fe
466258,ZIP220129113048148175,"Male,male,male,male",Proprietorship,Ma


In [19]:
df1.loc[
#     df16['Foreclosure_LOB'].notnull() &
      df1['left_2'].isin(['Fe']),  
#         df16['Micro_br'].isin([1]) & 
# # df16.loc[df16['Micro_br'].isin(['1']) & 
#         df16['schemedesc'].isin(['AUTO TOP UP PROG - SANJEEVANI SEC','SAATHIPLUS - SECURED','SAATHIPRIME - SECURED','SECURED - SANJEEVANI','SAHYOG SECURED - AIP','SAHYOG SECURED - ABB']),
        'First_female']=1

In [20]:
mask = ((df1['gender_value'].str.contains('Female')))
df1.loc[mask, 'Any_One_Female'] = 1

In [21]:
df1.columns

Index(['laf', 'gender_value', 'pan_constitution', 'left_2', 'First_female',
       'Any_One_Female'],
      dtype='object')

In [22]:
# df1.to_excel(r"C:\Users\saurabh.khopkar\Downloads\1.xlsx", index=False)

In [23]:
sql_query_2 = """

select laf,pan_no,buss_pan,gender_value from tb_bureau_data tbd 
where  edl_created_at >='2025-11-04'
order by gender_value 
"""


In [24]:
df2 = pd.read_sql_query(sql_query_2, connection)

In [25]:
# df2.columns = df2.columns.str.replace(' ', '')

In [26]:
df2['pan_no'] = df2['pan_no'].str.upper()
df2['buss_pan'] = df2['buss_pan'].str.upper()
df2['gender_value'] = df2['gender_value'].str.capitalize()

In [27]:
df2.loc[
    df2['buss_pan']==df2['pan_no'] ,
#     df2['gender_value'].isin(['Female']),  
#     df19['Foreclosure_LOB'].notnull(),
        'Pan_Match']=1

In [28]:
df2.loc[
    df2['Pan_Match'].isin([1]) &  
    df2['gender_value'].isin(['Female']),  
#     df19['Foreclosure_LOB'].notnull(),
        'Female_Pan_Match']=1

In [29]:
# df2.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df2.xlsx", index=False)

In [30]:
df3 = df2[(df2["Female_Pan_Match"]==1)]


In [31]:
df3 = df3.sort_values("laf").drop_duplicates(["laf"], keep="first")

In [32]:
neworder = ['laf','Female_Pan_Match']
df4=df3.reindex(columns=neworder)

In [33]:
df5=pd.merge(df1,df4,on='laf',how='left')

In [34]:
# df5.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df5.xlsx", index=False)

In [35]:


sql_query_6 = """
select 
lds.laf_number as laf,
cast(lds.datetime_login as date) as login_date,
b.agreementid,
b.agreementno, 
ds.fda as disb_amt,
ds.disb_date as first_disb_date,
b.schemedesc,
lds.program_name
from laf_detail_status lds 
left outer  join jayam_contract_details b on  b.laf_id=lds.laf_number 
left outer  join (select AGREEMENTID,min(DISB_DATE) as DISB_DATE ,SUM(DISB_AMT) as fda from jayam_disbursement_schedule
GROUP BY AGREEMENTID,STATUS) DS
on  b.agreementid =ds.agreementid
"""

In [36]:


df6 = pd.read_sql_query(sql_query_6, connection)

In [37]:
df6 = df6.sort_values("laf").drop_duplicates(["laf"], keep="first")

In [38]:
df6['first_disb_date'] = pd.to_datetime(df6['first_disb_date'])
df6['first_disb_Month'] = (df6['first_disb_date'] + pd.offsets.MonthEnd(0)).dt.date
df6['first_disb_Month'] = pd.to_datetime(df6['first_disb_Month'])

In [39]:
df7=pd.merge(df6,df5,on='laf',how='left')

In [40]:
df7.loc[
    df7['agreementno'].notnull(),
        'lan_count']=1


In [41]:
df7.loc[
#     df23['Foreclosure_LOB'].notnull() &
    df7['Female_Pan_Match'].isin([1]) & 
    df7['lan_count'].isin([1]) , 
#     df7['Pan_Match'].isin([1]) , 
#             ~df19['Intermidiate'].isin([1]) &     
#     df7['gender_value'].isin(['FEMALE','FeMale','female','FeMale','FEMale','Female']),  
#         df23['productflag'].isin(['SECLN']) & 
#         df23['C_scheme'].isin(['Sanjeevani']),
        'Female_PAN_Match_lan']=1

In [42]:
df7.columns

Index(['laf', 'login_date', 'agreementid', 'agreementno', 'disb_amt',
       'first_disb_date', 'schemedesc', 'program_name', 'first_disb_Month',
       'gender_value', 'pan_constitution', 'left_2', 'First_female',
       'Any_One_Female', 'Female_Pan_Match', 'lan_count',
       'Female_PAN_Match_lan'],
      dtype='object')

In [43]:
df7.loc[
#     df23['Foreclosure_LOB'].notnull() &
    df7['Any_One_Female'].isin([1]) & 
    df7['lan_count'].isin([1]) , 
#     df7['Pan_Match'].isin([1]) , 
#             ~df19['Intermidiate'].isin([1]) &     
#     df7['gender_value'].isin(['FEMALE','FeMale','female','FeMale','FEMale','Female']),  
#         df23['productflag'].isin(['SECLN']) & 
#         df23['C_scheme'].isin(['Sanjeevani']),
        'Any_One_Female_lan']=1

In [44]:
df7.columns

Index(['laf', 'login_date', 'agreementid', 'agreementno', 'disb_amt',
       'first_disb_date', 'schemedesc', 'program_name', 'first_disb_Month',
       'gender_value', 'pan_constitution', 'left_2', 'First_female',
       'Any_One_Female', 'Female_Pan_Match', 'lan_count',
       'Female_PAN_Match_lan', 'Any_One_Female_lan'],
      dtype='object')

In [45]:
# df7.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df7.xlsx", index=False)

In [46]:
df7['login_date'] = pd.to_datetime(df7['login_date'])
df7['login_date'] = pd.to_datetime(df7['login_date'])
df7['login_month'] = (df7['login_date'] + pd.offsets.MonthEnd(0)).dt.date
df7['login_month'] = pd.to_datetime(df7['login_month'])

In [47]:
# df7['login_date'] = pd.to_datetime(df7['login_date'])
# df7['login_date'] = pd.to_datetime(df7['login_date'])
# df7['login_month'] = (df7['login_date'] + pd.offsets.MonthEnd(0)).dt.date
# df7['login_month'] = pd.to_datetime(df7['login_month'])

In [48]:
# df7 = df7.fillna('')

In [49]:
df7.loc[
    df7['agreementno'].notnull(),
#      df1['lob'].isin(['Gro Micro Sales Secured','Sanjeevani Micro','Machinery Micro',
#                       'Rooftop Solar Micro','Gro Micro Sales Unsecured'])  ,
# #         df16['Micro_br'].isin([1]) & 
# # # df16.loc[df16['Micro_br'].isin(['1']) & 
#         df16['schemedesc_x'].isin(['GRO-MICRO-UNSECURED','SBL - UNSECURED','MICRO UNSECURED BLI']),
        'Disb_FY']=1

df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2018-04-01') & 
    (df7['first_disb_Month'] <= '2019-03-31') , 
    "18-19",df7['Disb_FY'])

df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2019-04-01') & 
    (df7['first_disb_Month'] <= '2020-03-31') , 
    "19-20",df7['Disb_FY'])

df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2020-04-01') & 
    (df7['first_disb_Month'] <= '2021-03-31') , 
    "20-21",df7['Disb_FY'])

df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2021-04-01') & 
    (df7['first_disb_Month'] <= '2022-03-31') , 
    "21-22",df7['Disb_FY'])

df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2022-04-01') & 
    (df7['first_disb_Month'] <= '2023-03-31') , 
    "22-23",df7['Disb_FY'])

df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2023-04-01') & 
    (df7['first_disb_Month'] <= '2024-03-31') , 
    "23-24",df7['Disb_FY'])

df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2024-04-01') & 
    (df7['first_disb_Month'] <= '2025-03-31') , 
    "24-25",df7['Disb_FY'])


df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2025-04-01') & 
    (df7['first_disb_Month'] <= '2026-03-31') , 
    "25-26",df7['Disb_FY'])


df7['Disb_FY'] = np.where(
    (df7['first_disb_Month'] >= '2026-04-01') & 
    (df7['first_disb_Month'] <= '2027-03-31') , 
    "26-27",df7['Disb_FY'])

In [50]:
# df7['Disb_FY'] = np.where(
#     (df7['disb_amt'] == "")  ,
# #     (df7['first_disb_Month'] <= '2025-03-31') , 
#     "NA",df7['Disb_FY'])
# df7['Disb_FY1'] = np.where(df7['first_disb_Month'].notnull(), df7['Disb_FY'],'NA')


In [51]:
df7.columns

Index(['laf', 'login_date', 'agreementid', 'agreementno', 'disb_amt',
       'first_disb_date', 'schemedesc', 'program_name', 'first_disb_Month',
       'gender_value', 'pan_constitution', 'left_2', 'First_female',
       'Any_One_Female', 'Female_Pan_Match', 'lan_count',
       'Female_PAN_Match_lan', 'Any_One_Female_lan', 'login_month', 'Disb_FY'],
      dtype='object')

In [52]:

df7= df7.drop(columns=['login_date','agreementno','schemedesc','program_name'])

In [53]:
# df7["laf_Length"]= df7["laf"].str.len() 

In [54]:
# df7 = df7.replace('NaN', 0)

In [55]:

df7.loc[
    df7['laf'].notnull(),
        'laf_count']=1


In [56]:
df7['disb_amt_Cr'] = df7['disb_amt']/10000000

In [57]:
df7['Female_PAN_Match_Amt'] = np.where(df7['Female_PAN_Match_lan'].notnull(), df7['disb_amt_Cr'],0)

In [58]:
df7['Any_One_Female_Amt'] = np.where(df7['Any_One_Female_lan'].notnull(), df7['disb_amt_Cr'],0)

In [59]:
# df7.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df7.xlsx", index=False)

In [60]:
df7.fillna(0, inplace=True)

In [61]:
print(df7['Disb_FY'].value_counts())

Disb_FY
nan      504948
23-24     43965
24-25     37345
22-23     29656
25-26     17738
21-22     13560
20-21       954
Name: count, dtype: int64


In [62]:
df7

,laf,agreementid,disb_amt,first_disb_date,first_disb_Month,gender_value,pan_constitution,left_2,First_female,Any_One_Female,Female_Pan_Match,lan_count,Female_PAN_Match_lan,Any_One_Female_lan,login_month,Disb_FY,laf_count,disb_amt_Cr,Female_PAN_Match_Amt,Any_One_Female_Amt
0,\nRUPITOL24092302300292772,0.0,0.0,0,0,"Female,female,female,female,female,female,fema...",Proprietorship,Fe,1.0,1.0,0.0,0.0,0.0,0.0,2024-09-30,nan,1.0,0.00,0.0,0.00
1,\nRUPITOL241011054500100189,0.0,0.0,0,0,"Female,female,female,female,male,male,male,male",Proprietorship,Fe,1.0,1.0,0.0,0.0,0.0,0.0,2024-10-31,nan,1.0,0.00,0.0,0.00
2,\rRUPITOL241014111500100476,0.0,0.0,0,0,"Female,female,female,female,male,male,male,male",Proprietorship,Fe,1.0,1.0,0.0,0.0,0.0,0.0,2024-10-31,nan,1.0,0.00,0.0,0.00
3,4FIN22041903450040945,0.0,0.0,0,0,"Female,female,female,female",Proprietorship,Fe,1.0,1.0,0.0,0.0,0.0,0.0,2022-04-30,nan,1.0,0.00,0.0,0.00
4,4FIN22050203550050840,254289.0,300000.0,2022-05-23 00:00:00,2022-05-31 00:00:00,"Female,female,female,female",Proprietorship,Fe,1.0,1.0,0.0,1.0,0.0,1.0,2022-05-31,22-23,1.0,0.03,0.0,0.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
648161,ZIP220129113034134166,0.0,0.0,0,0,"Female,female,female,female,male,male,male,male",Proprietorship,Fe,1.0,1.0,0.0,0.0,0.0,0.0,2022-01-31,nan,1.0,0.00,0.0,0.00
648162,ZIP220129113039139169,0.0,0.0,0,0,"Male,male,male,male,male,male,male,male",Proprietorship,Ma,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-31,nan,1.0,0.00,0.0,0.00
648163,ZIP220129113044144172,0.0,0.0,0,0,"Female,female,female,female,male,male,male,male",Proprietorship,Fe,1.0,1.0,0.0,0.0,0.0,0.0,2022-01-31,nan,1.0,0.00,0.0,0.00
648164,ZIP220129113048148175,0.0,0.0,0,0,"Male,male,male,male",Proprietorship,Ma,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-31,nan,1.0,0.00,0.0,0.00


In [63]:
df7.dtypes

laf                             object
agreementid                    float64
disb_amt                       float64
first_disb_date                 object
first_disb_Month                object
gender_value                    object
pan_constitution                object
left_2                          object
First_female                   float64
Any_One_Female                 float64
Female_Pan_Match               float64
lan_count                      float64
Female_PAN_Match_lan           float64
Any_One_Female_lan             float64
login_month             datetime64[ns]
Disb_FY                         object
laf_count                      float64
disb_amt_Cr                    float64
Female_PAN_Match_Amt           float64
Any_One_Female_Amt             float64
dtype: object

In [64]:
df8 = df7.pivot_table(values=['laf_count','Female_PAN_Match_lan','Any_One_Female_lan'], index=['Disb_FY'],
                        aggfunc="sum", fill_value=0, margins=True,margins_name= 'Total')

#                         aggfunc="sum", fill_value=0)
df8

,Any_One_Female_lan,Female_PAN_Match_lan,laf_count
Disb_FY,,,
20-21,542.0,73.0,954.0
21-22,9490.0,1339.0,13560.0
22-23,24126.0,2662.0,29656.0
23-24,31734.0,3140.0,43965.0
24-25,26856.0,3345.0,37345.0
25-26,8788.0,946.0,17738.0
nan,0.0,0.0,504948.0
Total,101536.0,11505.0,648166.0


In [65]:
# df8['Any_One_Female_lan%'] = df3[('LAN_Number','Total')]-df3[('LAN_Number','1. Current')]
df8['Any_One_Female_lan%'] = round(df8[('Any_One_Female_lan')]/df8[('laf_count')]*100)
df8['Female_PAN_Match_lan%'] = round(df8[('Female_PAN_Match_lan')]/df8[('laf_count')]*100,0)
df8

,Any_One_Female_lan,Female_PAN_Match_lan,laf_count,Any_One_Female_lan%,Female_PAN_Match_lan%
Disb_FY,,,,,
20-21,542.0,73.0,954.0,57.0,8.0
21-22,9490.0,1339.0,13560.0,70.0,10.0
22-23,24126.0,2662.0,29656.0,81.0,9.0
23-24,31734.0,3140.0,43965.0,72.0,7.0
24-25,26856.0,3345.0,37345.0,72.0,9.0
25-26,8788.0,946.0,17738.0,50.0,5.0
nan,0.0,0.0,504948.0,0.0,0.0
Total,101536.0,11505.0,648166.0,16.0,2.0


In [66]:
df9 = df7.pivot_table(values=['laf_count','disb_amt_Cr'], index=['Disb_FY'],
#                       df8 = df7.pivot_table(values=['laf','Female_PAN_Match_lan','Any_One_Female_lan'], index=['Disb_FY'], columns=['DPD_bucket'],
                        aggfunc="sum", fill_value=0, margins=True,margins_name= 'Total')
#                         aggfunc="sum", fill_value=0)
df9

,disb_amt_Cr,laf_count
Disb_FY,,
20-21,158.550046,954.0
21-22,1962.653979,13560.0
22-23,4330.832830,29656.0
23-24,5602.267901,43965.0
24-25,6935.804027,37345.0
25-26,4179.664526,17738.0
nan,0.000000,504948.0
Total,23169.773309,648166.0


In [67]:
df9 = df9.rename({"All_disb_amt_Cr":"disb_amt_Cr"})
df9

,disb_amt_Cr,laf_count
Disb_FY,,
20-21,158.550046,954.0
21-22,1962.653979,13560.0
22-23,4330.832830,29656.0
23-24,5602.267901,43965.0
24-25,6935.804027,37345.0
25-26,4179.664526,17738.0
nan,0.000000,504948.0
Total,23169.773309,648166.0


In [68]:
df10 = df7[(df7["Female_PAN_Match_lan"]==1)] #Female_PAN_Match_lan

In [69]:
df10A = df10.pivot_table(values=['laf_count','disb_amt_Cr'], index=['Disb_FY'],
#                       df8 = df7.pivot_table(values=['laf','Female_PAN_Match_lan','Any_One_Female_lan'], index=['Disb_FY'], columns=['DPD_bucket'],
                        aggfunc="sum", fill_value=0, margins=True,margins_name= 'Total')
#                         aggfunc="sum", fill_value=0)
df10A

,disb_amt_Cr,laf_count
Disb_FY,,
20-21,9.711123,73.0
21-22,192.614476,1339.0
22-23,457.033984,2662.0
23-24,555.095369,3140.0
24-25,631.867460,3345.0
25-26,242.338641,946.0
Total,2088.661053,11505.0


In [70]:
df11 = df7[(df7["Any_One_Female_lan"]==1)] #Female_PAN_Match_lan

In [71]:
df11A = df11.pivot_table(values=['laf_count','disb_amt_Cr'], index=['Disb_FY'],
#                       df8 = df7.pivot_table(values=['laf','Female_PAN_Match_lan','Any_One_Female_lan'], index=['Disb_FY'], columns=['DPD_bucket'],
                        aggfunc="sum", fill_value=0, margins=True,margins_name= 'Total')
#                         aggfunc="sum", fill_value=0)
df11A

,disb_amt_Cr,laf_count
Disb_FY,,
20-21,115.458279,542.0
21-22,1478.040374,9490.0
22-23,3398.729746,24126.0
23-24,4307.346304,31734.0
24-25,5470.686231,26856.0
25-26,2067.700438,8788.0
Total,16837.961373,101536.0


In [72]:
df12 = df7.pivot_table(values=['disb_amt_Cr','Female_PAN_Match_Amt','Any_One_Female_Amt'], index=['Disb_FY'],
#                       df8 = df7.pivot_table(values=['laf','Female_PAN_Match_lan','Any_One_Female_lan'], index=['Disb_FY'], columns=['DPD_bucket'],
                        aggfunc="sum", fill_value=0, margins=True,margins_name= 'Total')
#                         aggfunc="sum", fill_value=0)
df12

,Any_One_Female_Amt,Female_PAN_Match_Amt,disb_amt_Cr
Disb_FY,,,
20-21,115.458279,9.711123,158.550046
21-22,1478.040374,192.614476,1962.653979
22-23,3398.729746,457.033984,4330.832830
23-24,4307.346304,555.095369,5602.267901
24-25,5470.686231,631.867460,6935.804027
25-26,2067.700438,242.338641,4179.664526
nan,0.000000,0.000000,0.000000
Total,16837.961373,2088.661053,23169.773309


In [73]:
df12['Any_One_Female_amt%'] = round(df12[('Any_One_Female_Amt')]/df12[('disb_amt_Cr')]*100)
df12['Female_PAN_Match_Amt%'] = round(df12[('Female_PAN_Match_Amt')]/df12[('disb_amt_Cr')]*100,0)
df12

,Any_One_Female_Amt,Female_PAN_Match_Amt,disb_amt_Cr,Any_One_Female_amt%,Female_PAN_Match_Amt%
Disb_FY,,,,,
20-21,115.458279,9.711123,158.550046,73.0,6.0
21-22,1478.040374,192.614476,1962.653979,75.0,10.0
22-23,3398.729746,457.033984,4330.832830,78.0,11.0
23-24,4307.346304,555.095369,5602.267901,77.0,10.0
24-25,5470.686231,631.867460,6935.804027,79.0,9.0
25-26,2067.700438,242.338641,4179.664526,49.0,6.0
nan,0.000000,0.000000,0.000000,NaN,NaN
Total,16837.961373,2088.661053,23169.773309,73.0,9.0


In [74]:
# path0 = r"C:\Users\saurabh.khopkar\Downloads\Female_borrower ason 16th_may_25.xlsx"

# writer = pd.ExcelWriter(path0, engine = 'xlsxwriter')

# df7.to_excel(writer, sheet_name = "base_data", index = False)

# df8.to_excel(writer, sheet_name = "count_wit_perc")

# df9.to_excel(writer, sheet_name = "All_cases_count_nd_Amt")

# df10A.to_excel(writer, sheet_name = "Female_PAN_Match_lan")

# df11A.to_excel(writer, sheet_name = "Any_One_Female_lan")

# df12.to_excel(writer, sheet_name = "Amt_wit_perc")

In [75]:
file_path = 'C:\\Users\\saurabh.khopkar\\Downloads\\'
one_day_ago = datetime.now() - timedelta(days=1)
cur_time = one_day_ago.strftime("%d-%m-%Y_%H-%M-%S")
# cur_time = one_day_ago.strftime("%d-%m-%Y")
filename="Female_borrower_ason_" + str(cur_time) + ".xlsx"
# df.to_excel(file_path + filename, index=False)

In [76]:
writer = pd.ExcelWriter(file_path + filename, engine = 'xlsxwriter')

In [77]:
df7.to_excel(writer, sheet_name = "base_data", index = False)

In [ ]:
for column in df7:
    column_length = max(df7[column].astype(str).map(len).max(), len(column))
    col_idx = df7.columns.get_loc(column)
    writer.sheets['base_data'].set_column(col_idx, col_idx, column_length)

In [ ]:
df8.to_excel(writer, sheet_name = "count_wit_perc")

In [ ]:
for column in df8:
    column_length = max(df8[column].astype(str).map(len).max(), len(column))
    col_idx = df8.columns.get_loc(column)
    writer.sheets['count_wit_perc'].set_column(col_idx, col_idx, column_length)

In [ ]:
df9.to_excel(writer, sheet_name = "All_cases_count_nd_Amt")

In [ ]:
for column in df9:
    column_length = max(df9[column].astype(str).map(len).max(), len(column))
    col_idx = df9.columns.get_loc(column)
    writer.sheets['All_cases_count_nd_Amt'].set_column(col_idx, col_idx, column_length)

In [ ]:
df10A.to_excel(writer, sheet_name = "Female_PAN_Match_lan")

In [ ]:
for column in df10A:
    column_length = max(df10A[column].astype(str).map(len).max(), len(column))
    col_idx = df10A.columns.get_loc(column)
    writer.sheets['Female_PAN_Match_lan'].set_column(col_idx, col_idx, column_length)

In [ ]:
df11A.to_excel(writer, sheet_name = "Any_One_Female_lan")

In [ ]:
for column in df11A:
    column_length = max(df11A[column].astype(str).map(len).max(), len(column))
    col_idx = df11A.columns.get_loc(column)
    writer.sheets['Any_One_Female_lan'].set_column(col_idx, col_idx, column_length)

In [ ]:
df12.to_excel(writer, sheet_name = "Amt_wit_perc")

In [ ]:
for column in df12:
    column_length = max(df12[column].astype(str).map(len).max(), len(column))
    col_idx = df12.columns.get_loc(column)
    writer.sheets['Amt_wit_perc'].set_column(col_idx, col_idx, column_length)

In [ ]:
writer.close()